In [9]:
import os 
import numpy as np 
import pickle
import datetime as dt 

---

**Creating a trajectory dataset of the simulations**

In [1]:
NUM_SAMPLES_PER_TRAJECTORY = 10
SPLIT = 0.7

In [2]:
num_simulations = len([filename for filename in os.listdir("cuboid_simulations") if ".npy" in filename])
num_train_simulations = int(SPLIT*num_simulations)
print(num_simulations, "total simulations")
print(num_train_simulations, "training simulations")

1000 total simulations
700 training simulations


In [7]:
V = []
for filename in os.listdir("cuboid_simulations"):
    if ".npy" in filename:
        v = np.load(open("cuboid_simulations/" + filename, "rb"))
        # 500+ samples for one item's trajectory in one particular simulation is probably too dense
        # I will sample NUM_SAMPLES_PER_TRAJECTORY entries, and try to have enough simulations and items 
        # to have a large and diverse dataset
        v = v[np.random.choice(range(len(v)), size=min(NUM_SAMPLES_PER_TRAJECTORY, len(v)), replace=False)]
        # Here I don't care about different timesteps. I simply want a list of 11-vectors
        # TODO: be able to deal with examples of varying number of items and timesteps
        #       e.g. by filling them up with "null items"
        #       but what would be even better (though I'm not sure how to do this yet), is to actually have them be empty
        #       and then later let the model learn an embedding corresponding to an [EMPTY] token
        #
        #
        #
        #
        V.append(v)

V_train = np.concatenate(V[:num_train_simulations])
V_test = np.concatenate(V[num_train_simulations:]) 

# Normalize the data using statistics from the training set
mean, std = V_train.mean(), V_train.std()
V_train = (V_train-mean)/std
V_test = (V_test-mean)/std

# No longer needed
del V, mean, std

In [12]:
# Store the whole dataset (before applying embeddings since I'm just using a random linear projection here for experimentation) 
# as a single pickled file

timestamp = str(dt.datetime.now()).replace(" ", "_").replace(":", "_").replace("-", "_").replace(".", "_")
pickle.dump((V_train, V_test), open(f"datasets/trajectories/dataset_{timestamp}.pickle", "wb"))

# E.g. in Colab, read it like this:
# V_train, V_test = pickle.load(open("datasets/trajectories/dataset_2023_01_18_17_06_06_959337.pickle", "rb"))

In [14]:
V_train.shape

(7000, 10, 11)